# Predicting base-Kissat runtime parameters using AIG representations

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import GCNConv
import torch_geometric.nn as gnn

In [ ]:
class Predictor(nn.Module):
	def __init__(self, hidden_dim=64, synvec_len=20):
		super().__init__()

		# =========================
		# GCN Layer Declaration
		# =========================
		self.gcn1 = gnn.GCNConv(4, hidden_dim)
		self.gcn2 = gnn.GCNConv(hidden_dim, hidden_dim)
		
	def forward(self, data):
		"""
		data.batch must exist if using DataLoader
		"""

		# =========================
		# Build Node Features
		# =========================

		# node_type → one-hot
		node_type = F.one_hot(
			data.node_type.long(),
			num_classes=3
		).float()

		# num_inverted_predecessors
		inv_pred = data.num_inverted_predecessors.view(-1, 1).float() # since node_type → shape (num_nodes, 3) but inv_pred is (num_nodes,) → reshape to (num_nodes, 1) for concatenation

		# concatenate → (num_nodes, 4)
		x = torch.cat([node_type, inv_pred], dim=1)

		# =========================
		# GCN Forward
		# =========================

		x = F.relu(self.gcn1(x, data.edge_index))
		x = F.relu(self.gcn2(x, data.edge_index))

		# graph-level embedding
		x = gnn.pool.global_mean_pool(x, data.batch) #shape after: [batch_size, hidden_dim]
		return x
